# MatrixWorld: Upper Triangularisation via Reinforcement Learning

This notebook frames a linear algebra problem as a **Reinforcement Learning environment**: given a random matrix $n \times n$ over $\mathbb{F}_2$, the goal is to make it **upper triangular** (i.e. all entries below the main diagonal equal to zero) using only two types of elementary row operations.

Two approaches are compared:
1. **Q-Learning** 
2. **DQN**

# Approach 1: Q-learning

In [ ]:
import numpy as np
import gymnasium as gym
from collections import defaultdict
import sys, time

A $n \times n$ matrix $A$ over $\mathbb{F}_2$ is upper triangular if and onyl if
$$ A_{ij}=0 \quad \text{per ogni } i>j. $$

One of the most natural ways to quantify how far a matrix is from this form is to count how many `1` entries there are **strictly below the diagonal**.
We introduce the function **`how_close`** to this aim : The value `0` means the matrix is alreadu upper triangular.

In [ ]:
def how_close(M:np.ndarray):
    n = M.shape[0]
    return int(sum(M[i][j] for i in range(n) for j in range(i)))

## The Environment: `MatrixWorldEnv`


### State Space
The **observation** is a binary $n \times n$ matrix (with values in $\mathbb{F}_2$, i.e. $\{0,1\}$), randomly taken at the beginning of each episode. The initial state always contains at least one `1` below the diagonal (otherwise the problem would already be solved).

### Action Space
The agent can perform two types of **elementary row operations** (the same operations used in Gaussian elimination):

| Type | Description | Number of actions ($n=4$) |
|------|-------------|---------------------------|
| **Swapping rows** `k=0` | Swap rows $i$ and $j$ (with $i < j$) | $\binom{n}{2}$ |
| **Transvection** `k=1` | Replace row $i$ with $i + j \pmod{2}$ | $n(n-1)$ |

All operations are performed over $\mathbb{F}_2$ (**modulo 2**).

### Reward
The reward structure guides the agent toward an upper triangular form:

| Situation | Reward |
|-----------|--------|
| Upper triangular matrix (terminated) | `+100` |
| Step limit exceeded (truncated) | `−100` |
| The number of `1`s below the diagonal decreases | `previous_distance − new_distance` (> 0) |
| No progress | `−10` |

In this example we will stop the episode after $2n^3$ steps if the matrix is not upper triangular after these steps.

In [ ]:
encode_obs = lambda obs: tuple(obs.flatten())
class MatrixWorldEnv(gym.Env):
    def __init__(self, size):
        self.size = size
        self.observation_space = gym.spaces.Box(0, 1, shape=(size,size), dtype=np.int8)

        self._matrix = np.zeros((size,size), dtype=np.int8)
        self._list_to_action = [
            (i, j, 0) for i in range(size) for j in range(i + 1, size)
        ] 
        self._list_to_action += [
            (i, j, 1) for i in range(size) for j in range(size) if i != j
        ]
        self._steps = 0

        self.action_space = gym.spaces.Discrete(len(self._list_to_action))
        self.render_mode = "ansi"

    def _get_obs(self):
        return self._matrix

    def _get_info(self):
        """Compute auxiliary information for debugging.
        Returns:
        dict: Info with distance between agent and target
        """
        return {'distance': how_close(self._matrix)}

    def reset(self, seed=None, options=None):
        """Start a new episode.
        seed: Random seed for reproducible episodes
        options: Additional configuration (unused in this example)

        Returns:
        tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        self._matrix = self.observation_space.sample()
        while how_close(self._matrix) == 0:
            self._matrix = self.observation_space.sample()
        self._steps = 0

        observation = self._get_obs()
        info = self._get_info()
        return observation, info
	
    def step(self, action):

        distance = how_close(self._matrix)

        (i, j, k) = self._list_to_action[action]
        self._steps += 1
        if k == 0: 
            # trasposition
            self._matrix[[i, j]] = self._matrix[[j, i]]
        else:
            # transvection
            self._matrix[i] += self._matrix[j]
        self._matrix = self._matrix % 2

        newdistance = how_close(self._matrix)

	    # Check if agent reached the target
        terminated = how_close(self._matrix) == 0

        # We use truncation in this environment
        truncated = self._steps > 2*self.size**3
        #truncated = False

        # Simple reward structure: 
        # +100 for reaching upper triangular matrix
        # -100 if truncated 
        # -10 if there are no changes
        # distance - newdistance if the agend does some change, we reward if it is going 'to the upper trinagular form'
        # Alternative: could give small negative rewards for each step to
        # encourage efficiency
        if terminated:
            # Finishes in upper triangular form
            reward = 100 
        elif truncated:
            # Finishes without upper triagular form
            reward = -100
        elif distance != newdistance:
            reward = distance - newdistance
        else: 
            reward = -10


        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

    def render(self):
        return f"{self._matrix}"

## Tabular Q-Learning (the `Gauss` Agent)

### Idea
Tabular Q-learning maintains a **lookup table `q_values`** that maps each (state, action) pair to a scalar estimating the expected future reward. The table is updated at every step via the following equation:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

where $\alpha$ is the learning rate and $\gamma$ the discount factor.

### Exploration: $\varepsilon$-greedy policy
- With probability $\varepsilon$: pick a **random action** (exploration)  
- With probability $1 - \varepsilon$: pick the action with the highest Q-value (exploitation)  
- $\varepsilon$ decays gradually over the course of training


### Limitations
The state space grows exponentially with $n$ (each distinct binary $n \times n$ matrix is a separate state). This approach is therefore practical only for small matrices (typically $n \leq 4$).

In [ ]:
class Gauss:
    def __init__(self, env, learning_rate, initial_epsilon, epsilon_decay,
                 final_epsilon, discount_factor=0.99):
        """Initialize a Reinforcement Learning agent with an empty dictionary
        of state-action values (q_values), a learning rate and an epsilon.

        Args:
            learning_rate: The learning rate
            initial_epsilon: The initial epsilon value
            epsilon_decay: The decay for epsilon
            final_epsilon: The final epsilon value
            discount_factor: The discount factor for computing the Q-value
        """
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor

        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        self.training_error = []

    def get_action(self, env, obs):
        """
        Returns the best action with probability (1 - epsilon)
        otherwise a random action with probability epsilon to ensure exploration.
        """
        # with probability epsilon return a random action to explore the environment
        if np.random.random() < self.epsilon:
            return env.action_space.sample()
        # with probability (1 - epsilon) act greedily (exploit)
        q = self.q_values[encode_obs(obs)]
        max_q = np.max(q)
        best_actions = np.flatnonzero(np.isclose(q, max_q))
        return int(np.random.choice(best_actions))
    
    def update(self, obs, action, reward, terminated, next_obs):
        """Updates the Q-value of an action."""

        s = encode_obs(obs)
        s_next = encode_obs(next_obs)
        future_q_value = (not terminated) * np.max(self.q_values[s_next])
        temporal_difference = (
            reward + self.discount_factor * future_q_value - self.q_values[s][action]
        )

        self.q_values[s][action] = (
            self.q_values[s][action] + self.lr * temporal_difference
        )
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)

In [ ]:
from IPython.display import display, clear_output

def set_cursor(state):
    sys.stdout.write("\x1b[?25h" if state else "\x1b[?25l")

def clear_screen():
    sys.stdout.write("\x1b[2J\x1b[H")

def print_state(header, body):
    clear_screen()
    sys.stdout.write(header + '\n')
    sys.stdout.write(env.render())

def playtrain(render=False):
    obs, info = env.reset()
    init_obs = obs.copy()
    done = False
    cnt = 0
    if render:
        set_cursor(False)
    while not done:
        action = agent.get_action(env, obs)
        if render:
            clear_output(wait=True)        
            print_state(f'Move #{cnt}', env.render())
            sys.stdout.flush()
        next_obs, reward, terminated, truncated, info = env.step(action)
        # update the agent
        agent.update(obs, action, reward, terminated, next_obs)
        if render:
            sys.stdout.write(f"\nNumber of entries: {info['distance']:.2f}\n")
            sys.stdout.write(f"\nReward: {reward:.2f}\n")
            sys.stdout.flush()
            time.sleep(1)
        # update if the environment is done and the current obs
        done = terminated or truncated
        obs = next_obs
        cnt += 1
    agent.decay_epsilon()
    if render:
        clear_output(wait=True)            
        clear_screen()
        print(f'Original state:\n{init_obs}\nFinal state:\n{env.render()}\nAfter {cnt} moves')
        set_cursor(True)
        

def train_Gauss(agent:Gauss, n_episodes=50_000):
    for episode in range(n_episodes):
        if not (episode % 100):
            clear_screen()
            print(f'EPISODE {episode} (eps = {agent.epsilon:.2f}, training error = {agent.training_error[-1] if episode else 0:.2f})')
        playtrain()
    print("Finished training.")

## Training: Q-Learning on MatrixWorldEnv(3)

We create the environment with $n=3$ and train the agent for **150,000 episodes**.

Key hyperparameters:
- `learning_rate = 0.1` — how much each Q-table update is weighted  
- `epsilon_decay` — $\varepsilon$ decreases from 1 to 0.1 over the first half of training  
- `discount_factor = 0.99` (default) — nearly equally weights immediate and future rewards

In [ ]:
env = MatrixWorldEnv(3)

# hyperparameters
learning_rate = 0.1
start_epsilon = 1
epsilon_decay = start_epsilon / (100_000 / 2) 
final_epsilon = 0.1

agent = Gauss(
    env=env,
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

train_Gauss(agent, n_episodes=150000)

## Evaluating the Q-Learning Agent

After training, we set $\varepsilon = 0$ (no exploration) and let the agent run one episode.

In [ ]:
agent.epsilon = 0
playtrain(render=True)

# Questions
Think about these before moving on:

1. How many episodes are needed before the agent reliably find the upper triangular form?
2. How does changing the hyperparameters affect the behviour of the learning? 
3. How does truncation affect the behviour of the learning? 
4. How does the size of the matrix affect training time and result?

---

# Approach 2: DQN

### Motivation
Tabular Q-learning does not scale: for $n=4$ the state space has $2^{16} = 65{,}536$ possible matrices, and for $n=5$ already over 33 million. A **neural network** can approximate the Q-function without storing every state.

### DQN (Deep Q-Network)
**DQN** replaces the table with a neural network $Q_\theta(s, a)$. The network is trained by minimising the Bellman loss on minibatches sampled from an **experience replay buffer**, which improves stability and sample efficiency.

We use the **Stable Baselines3** implementation, which handles automatically:
- Replay buffer and target network updates
- $\varepsilon$-greedy exploration with decay schedule
- Network optimisation via Adam

### Comparison with the tabular approach
| Feature | Tabular Q-Learning | DQN (SB3) |
|---|---|---|
| Q representation | Python dictionary | Neural network |
| Scalability | Small $n$ only | Larger matrices |
| Training time | Fast for $n=3$ | Slower but more powerful |

In [ ]:
import numpy as np
import gymnasium as gym
from collections import defaultdict
import sys, time
from stable_baselines3 import DQN
from stable_baselines3 import PPO

### Environment for DQN

The environment is the same `MatrixWorldEnv`. The reward logic is identical.

In [ ]:
encode_obs = lambda obs: tuple(obs.flatten())

def how_close(M:np.ndarray):
    n = M.shape[0]
    return int(sum(M[i][j]*(i*n+j) for i in range(n) for j in range(i)))

In this example we add an extra column to the observation matrix to give the agent information about the number of steps. It is helpful to avoid cycles. 
This is a matrix in the observation space. We are not going to perform computation on this matrix,the operations are performed on `self._matrix`.

In [ ]:
class MatrixWorldEnv(gym.Env):
    def __init__(self, size):
        self.size = size
        
        # --- CHANGE 1: Expand obs space by +1 column for step count ---
        self.observation_space = gym.spaces.Box(
            0, 1, shape=(size, size + 1), dtype=np.float32
        )

        self._matrix = np.zeros((size, size), dtype=np.int8)
        self._list_to_action = [
            (i, j, 0) for i in range(size) for j in range(i + 1, size)
        ]
        self._list_to_action += [
            (i, j, 1) for i in range(size) for j in range(size) if i != j
        ]
        self._steps = 0

        self.action_space = gym.spaces.Discrete(len(self._list_to_action))
        self.render_mode = "ansi"

    # --- CHANGE 2: Include the step counter in the observation ---
    def _get_obs(self):
        obs = np.zeros((self.size, self.size + 1), dtype=np.float32)
        obs[:, :self.size] = self._matrix

        # normalize step count (optional, but recommended)
        obs[:, -1] = self._steps / (2 * self.size**3)

        return obs

    def _get_info(self):
        return {'distance': how_close(self._matrix)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self._matrix = self._matrix = np.random.randint(0, 2, size=(self.size, self.size))
        while how_close(self._matrix) == 0:
            self._matrix = np.random.randint(0, 2, size=(self.size, self.size))

        self._steps = 0

        observation = self._get_obs()
        info = self._get_info()
        return observation, info

    def step(self, action):
        distance = how_close(self._matrix)
        (i, j, k) = self._list_to_action[action]
        self._steps += 1

        if k == 0:
            self._matrix[[i, j]] = self._matrix[[j, i]]
        else:
            self._matrix[i] += self._matrix[j]

        self._matrix = self._matrix % 2

        newdistance = how_close(self._matrix)
        terminated = how_close(self._matrix) == 0
        #truncated = False
        truncated = self._steps > 20# 2 * self.size**3
        
        reward = -0.1

        observation = self._get_obs()
        info = self._get_info()
        return observation, reward, terminated, truncated, info

    def render(self):
        return f"{self._matrix}"

### `Gauss` Class 

The `Gauss` class and rendering utilities are redefined here for completeness. The actual training is handled by SB3 DQN. The `play_sb` function is the SB3 equivalent of `playtrain`.

In [ ]:
class Gauss:
    def __init__(self, env, learning_rate, initial_epsilon, epsilon_decay,
                 final_epsilon, discount_factor=0.99):
        """Initialize a Reinforcement Learning agent with an empty dictionary
        of state-action values (q_values), a learning rate and an epsilon.

        Args:
            learning_rate: The learning rate
            initial_epsilon: The initial epsilon value
            epsilon_decay: The decay for epsilon
            final_epsilon: The final epsilon value
            discount_factor: The discount factor for computing the Q-value
        """
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor

        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        self.training_error = []

    def get_action(self, env, obs):
        """
        Returns the best action with probability (1 - epsilon)
        otherwise a random action with probability epsilon to ensure exploration.
        """
        # with probability epsilon return a random action to explore the environment
        if np.random.random() < self.epsilon:
            return env.action_space.sample()
        # with probability (1 - epsilon) act greedily (exploit)
        q = self.q_values[encode_obs(obs)]
        max_q = np.max(q)
        best_actions = np.flatnonzero(np.isclose(q, max_q))
        return int(np.random.choice(best_actions))

    def update(self, obs, action, reward, terminated, next_obs):
        """Updates the Q-value of an action."""

        s = encode_obs(obs)
        s_next = encode_obs(next_obs)
        future_q_value = (not terminated) * np.max(self.q_values[s_next])
        temporal_difference = (
            reward + self.discount_factor * future_q_value - self.q_values[s][action]
        )

        self.q_values[s][action] = (
            self.q_values[s][action] + self.lr * temporal_difference
        )
        self.training_error.append(temporal_difference)

    def decay_epsilon(self):
        self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)
        

def set_cursor(state):
    sys.stdout.write("\x1b[?25h" if state else "\x1b[?25l")

def clear_screen():
    sys.stdout.write("\x1b[2J\x1b[H")

def print_state(header, body):
    clear_screen()
    sys.stdout.write(header + '\n')
    sys.stdout.write(env.render())
    

def play_sb(render=False):
    obs, info = env.reset()
    init_obs = obs.copy()
    done = False
    cnt = 0
    if render:
        set_cursor(False)
    while not done:
        action, _ = model.predict(obs, deterministic=False)
        next_obs, reward, terminated, truncated, info = env.step(action)  # prima lo step
        if render:
            clear_output(wait=True)
            print_state(f'Move #{cnt}', env.render())
            sys.stdout.write(f"\nNumber of entries: {info['distance']:.2f}\n")
            sys.stdout.write(f"\nReward: {reward:.2f}\n")
            sys.stdout.flush()
            time.sleep(0.5)
        done = terminated or truncated
        obs = next_obs
        cnt += 1
    if render:
        clear_output(wait=True)
        clear_screen()
        print(f'Original state:\n{init_obs}\nFinal state:\n{env.render()}\nAfter {cnt} moves')
        set_cursor(True)
    return cnt, init_obs


In [ ]:
env = MatrixWorldEnv(4)

obs, info = env.reset()
policy_kwargs = dict(
    net_arch=[64, 32]  # hidden layers
)

model = DQN("MlpPolicy", env, 
            policy_kwargs=policy_kwargs,
            verbose=1000,learning_rate=0.00001, 
            exploration_fraction=0.5,
            exploration_final_eps=0.01,
            train_freq=8)
model.learn(total_timesteps=500_000)


play_sb(render=False)


L =[]
L = [play_sb(render=False) for _ in range(1000)]
k = max(a for a, _ in L)
print(next(a for i,a in L if i ==k), k)

print(model.policy.q_net)

In [ ]:
play_sb(True)

# Adding a column in Q-learning
Try the same approach (extra column for steps) with Q-learning.

In [ ]:
# Your code.